# Data_Eng Homework 2 
## Maya Westra
## May 3 2026

In [1]:
#!pip install datarec-lib 

## Step 1 - Download and Store Dataset in S3

In this step we check:
- ensure we have the MovieLens dataset downloaded
- check whether the MovieLens dataset already exists in the S3 bucket,
- avoid duplicate downloads,
- and upload the dataset only if necessary.

In [4]:
import boto3
from botocore.exceptions import ClientError

BUCKET = "maya-westra-de300-hw4"
REGION = "us-east-1"

s3 = boto3.client("s3", region_name=REGION)

In [7]:

def dataset_exists():
    # Check whether the MovieLens dataset already exists in the S3 bucket.
    
    try:
        # check if the dataset file exists in S3
        s3.head_object(Bucket=BUCKET,  Key="ml-1m.zip")

        print("Dataset already exists in S3.")
        return True

    except ClientError:
        print("Dataset does not exist in S3.")
        return False


dataset_exists()

Dataset already exists in S3.


True

## Step 2 — Create BERT Embeddings for Movies Released Before 1980

In this step, I generate BERT embeddings for movies released in or before 1980. This represents the offline stage of the recommendation pipeline, where embeddings are computed once and stored for later recommendation generation.

The workflow:

1. Import necesary packages 
2. Download the dataset from S3,
3. Extract the MovieLens files,
4. Filter movies released before 1980,
5. Create sentence embeddings using SentenceTransformers,
6. Saves the intermediate outputs back to S3.

In [8]:
#!pip install sentence-transformers

In [9]:
import os
import zipfile
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

BUCKET = "maya-westra-de300-hw4"
s3 = boto3.client("s3")

In [15]:
def download_dataset_from_s3():
    s3.download_file(BUCKET, "ml-1m.zip", "ml-1m.zip")
    print("Downloaded ml-1m.zip from S3")


def unzip_dataset():
    if not os.path.exists("ml-1m"):
        with zipfile.ZipFile("ml-1m.zip", "r") as z:
            z.extractall(".")
        print("Unzipped dataset")
    else:
        print("Dataset already unzipped")

In [16]:
def load_movies_before_1980():
    movies = pd.read_csv(
        "ml-1m/movies.dat",
        sep="::",
        engine="python",
        encoding="latin-1",
        names=["MovieID", "Title", "Genres"]
    )

    movies["Year"] = movies["Title"].str.extract(r"\((\d{4})\)").astype(float)
    movies_1980 = movies[movies["Year"] <= 1980].copy()

    movies_1980["Text"] = (
        movies_1980["Title"].fillna("") 
        + " Genres: " 
        + movies_1980["Genres"].fillna("")
    )

    print("Movies before/equal 1980:", len(movies_1980))
    return movies_1980

In [17]:
def create_bert_embeddings(movies_df):
    model = SentenceTransformer("all-MiniLM-L6-v2")

    texts = movies_df["Text"].tolist()

    embeddings = model.encode(
        texts,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    print("Embeddings shape:", embeddings.shape)
    return embeddings

In [18]:
def save_embeddings_to_s3(movies_df, embeddings):
    movies_df.to_csv("movies_before_1980.csv", index=False)
    np.save("movie_embeddings_before_1980.npy", embeddings)

    s3.upload_file(
        "movies_before_1980.csv",
        BUCKET,
        "intermediate/movies_before_1980.csv"
    )

    s3.upload_file(
        "movie_embeddings_before_1980.npy",
        BUCKET,
        "intermediate/movie_embeddings_before_1980.npy"
    )

    print("Saved intermediate files to S3")

In [19]:
def run_offline_embedding_step():
    download_dataset_from_s3()
    unzip_dataset()

    movies_1980 = load_movies_before_1980()
    embeddings = create_bert_embeddings(movies_1980)

    save_embeddings_to_s3(movies_1980, embeddings)

    return movies_1980, embeddings

In [20]:
movies_1980, embeddings_1980 = run_offline_embedding_step()

Downloaded ml-1m.zip from S3
Dataset already unzipped
Movies before/equal 1980: 887


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Embeddings shape: (887, 384)
Saved intermediate files to S3


**The offline embedding pipeline completed successfully.** A total of 887 movies released in or before 1980 were processed, and 384-dimensional BERT embeddings were generated using the `all-MiniLM-L6-v2` model. The resulting intermediate files were saved to the S3 bucket for later recommendation generation.

## Step 3 — Recommend Movies for Cold User and Top User

In this step, I use the pre-1980 movie embeddings created in Step 2 to generate recommendations for two user types. 

- For the cold user, there is no interaction history, so I recommend the most commonly rated movies from the candidate set. 

- For the top user, I select a random user from the top 5% of users by number of ratings, create a user embedding by averaging the embeddings of movies they rated 4 or higher, and recommend movies with the highest cosine similarity that the user has not already seen.

### 3a) Load Pre-1980 Movies and Embeddings

First, I load the movie dataset and embeddings generated in Step 2 from S3. I display the first few rows, to contectualize the dataset 

In [21]:
import random
from sklearn.metrics.pairwise import cosine_similarity

BUCKET = "maya-westra-de300-hw4"
s3 = boto3.client("s3")

In [22]:
s3.download_file(BUCKET, "intermediate/movies_before_1980.csv", "movies_before_1980.csv")
s3.download_file(BUCKET, "intermediate/movie_embeddings_before_1980.npy", "movie_embeddings_before_1980.npy")

movies = pd.read_csv("movies_before_1980.csv")
embeddings = np.load("movie_embeddings_before_1980.npy")

movies.head()

,MovieID,Title,Genres,Year,Text
0,111,Taxi Driver (1976),Drama|Thriller,1976.0,Taxi Driver (1976) Genres: Drama|Thriller
1,154,Belle de jour (1967),Drama,1967.0,Belle de jour (1967) Genres: Drama
2,199,"Umbrellas of Cherbourg, The (Parapluies de Che...",Drama|Musical,1964.0,"Umbrellas of Cherbourg, The (Parapluies de Che..."
3,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Fantasy|Sci-Fi,1977.0,Star Wars: Episode IV - A New Hope (1977) Genr...
4,390,Faster Pussycat! Kill! Kill! (1965),Action|Comedy|Drama,1965.0,Faster Pussycat! Kill! Kill! (1965) Genres: Ac...


### 3b) Load User Ratings

Next, I load the MovieLens ratings data, which will be used to identify popular movies and construct user preference profiles. I also display the first few rows to better understand how the information is structured. 

In [23]:
s3.download_file(BUCKET, "ml-1m.zip", "ml-1m.zip")

if not os.path.exists("ml-1m"):
    with zipfile.ZipFile("ml-1m.zip", "r") as z:
        z.extractall(".")

ratings = pd.read_csv(
    "ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["UserID", "MovieID", "Rating", "Timestamp"]
)

ratings.head()

,UserID,MovieID,Rating,Timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


### 3c) Identify Top User 

This section identifies a random top user from the top 5% of users by number of ratings. 

In [28]:
user_counts = ratings.groupby("UserID").size()
cutoff = user_counts.quantile(0.95)

top_user_ids = user_counts[user_counts >= cutoff].index.tolist()
top_user_id = random.choice(top_user_ids)

print("Top user:", top_user_id)
print("Number of ratings:", user_counts[top_user_id])

Top user: 1264
Number of ratings: 711


### 3d) Defines two recommendation functions: 
- One function is for a cold user using movie popularity,
- One function is for a top user. It uses cosine similarity between the user’s liked movie embeddings and all candidate movie embeddings.

In [29]:
def recommend_cold_user(n=5):
    movie_counts = ratings.groupby("MovieID").size().reset_index(name="NumRatings")
    candidates = movies.merge(movie_counts, on="MovieID", how="left")
    candidates["NumRatings"] = candidates["NumRatings"].fillna(0)

    recs = candidates.sort_values("NumRatings", ascending=False).head(n)
    return recs["Title"].tolist()


def recommend_top_user(user_id, n=5):
    user_ratings = ratings[ratings["UserID"] == user_id]

    liked_ids = user_ratings[user_ratings["Rating"] >= 4]["MovieID"].tolist()
    liked_movies = movies[movies["MovieID"].isin(liked_ids)]

    liked_indices = liked_movies.index.tolist()
    user_embedding = embeddings[liked_indices].mean(axis=0).reshape(1, -1)

    scores = cosine_similarity(user_embedding, embeddings)[0]

    movies_scored = movies.copy()
    movies_scored["score"] = scores

    seen_ids = set(user_ratings["MovieID"])
    movies_scored = movies_scored[~movies_scored["MovieID"].isin(seen_ids)]

    recs = movies_scored.sort_values("score", ascending=False).head(n)
    return recs["Title"].tolist()

In [31]:
cold_recs = recommend_cold_user()
top_recs = recommend_top_user(top_user_id)

last_interaction_time = pd.to_datetime(
    ratings[ratings["UserID"] == top_user_id]["Timestamp"].max(),
    unit="s"
)

recommendations = pd.DataFrame([
    {
        "User_Type": "Cold User",
        "Last_Interaction_Time": None,
        "User_Summary": "No previous interaction data available.",
        "Recommended_Movies": cold_recs
    },
    {
        "User_Type": "Top User",
        "Last_Interaction_Time": last_interaction_time,
        "User_Summary": f"UserID {top_user_id}; total ratings = {user_counts[top_user_id]}; selected from top 5% of users by number of interactions.",
        "Recommended_Movies": top_recs
    }
])

recommendations.to_csv(
    "recommendations_before_1980.csv",
    index=False
)

s3.upload_file(
    "recommendations_before_1980.csv",
    BUCKET,
    "outputs/recommendations_before_1980.csv"
)

recommendations

,User_Type,Last_Interaction_Time,User_Summary,Recommended_Movies
0,Cold User,NaT,No previous interaction data available.,"[Star Wars: Episode IV - A New Hope (1977), St..."
1,Top User,2000-12-03 23:19:43,UserID 1264; total ratings = 711; selected fro...,"[Very Natural Thing, A (1974), Windows (1980),..."


The recommendation results were successfully generated and uploaded to S3. They are also displayed above for reference. 

## Step 4 — Repeat Workflow Using Full MovieLens Dataset

In this step, I repeat the embedding and recommendation workflow using the full MovieLens dataset instead of only movies released before 1980. The same recommendation logic is reused, but embeddings are generated for all movies in the dataset.

### 4a) Generate Embeddings for Full Dataset

This section loads all movies from the MovieLens dataset, creates BERT embeddings for every movie, and uploads the resulting files to S3 for later recommendation generation.

In [32]:
def load_all_movies():
    movies = pd.read_csv(
        "ml-1m/movies.dat",
        sep="::",
        engine="python",
        encoding="latin-1",
        names=["MovieID", "Title", "Genres"]
    )

    movies["Text"] = movies["Title"].fillna("") + " Genres: " + movies["Genres"].fillna("")
    return movies

In [35]:
all_movies = load_all_movies()
all_embeddings = create_bert_embeddings(all_movies)

all_movies.to_csv("movies_all.csv", index=False)
np.save("movie_embeddings_all.npy", all_embeddings)

s3.upload_file("movies_all.csv", BUCKET, "intermediate/movies_all.csv")
s3.upload_file("movie_embeddings_all.npy", BUCKET, "intermediate/movie_embeddings_all.npy")
print("Saved full dataset embeddings to S3.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/122 [00:00<?, ?it/s]

Embeddings shape: (3883, 384)
Saved full dataset embeddings to S3.


### 4b) Generate Full-Dataset Recommendations

This section generates recommendations using embeddings created from the full MovieLens dataset. The cold user and top user reccomendations were made using the same method as in step 3

In [36]:
user_counts = ratings.groupby("UserID").size()
cutoff = user_counts.quantile(0.95)

top_user_ids = user_counts[user_counts >= cutoff].index.tolist()
top_user_id = random.choice(top_user_ids)

cold_recs = recommend_cold_user()
top_recs = recommend_top_user(top_user_id)

last_interaction_time = ratings[ratings["UserID"] == top_user_id]["Timestamp"].max()

recommendations_all = pd.DataFrame([
    {
        "User_Type": "Cold User",
        "Last_Interaction_Time": None,
        "User_Summary": "No previous interaction data available.",
        "Recommended_Movies": cold_recs
    },
    {
        "User_Type": "Top User",
        "Last_Interaction_Time": pd.to_datetime(last_interaction_time, unit="s"),
        "User_Summary": f"UserID {top_user_id}; total ratings = {user_counts[top_user_id]}; selected from top 5% of users by number of interactions.",
        "Recommended_Movies": top_recs
    }
])

recommendations_all.to_csv("recommendations_all.csv", index=False)

s3.upload_file(
    "recommendations_all.csv",
    BUCKET,
    "outputs/recommendations_all.csv"
)

recommendations_all

,User_Type,Last_Interaction_Time,User_Summary,Recommended_Movies
0,Cold User,NaT,No previous interaction data available.,"[Star Wars: Episode IV - A New Hope (1977), St..."
1,Top User,2002-11-24 16:23:52,UserID 2092; total ratings = 729; selected fro...,"[Limelight (1952), Very Natural Thing, A (1974..."


The recommendation results were successfully generated and uploaded to S3. They are also displayed above for reference. 

## Step 5 — Create Personal User Profile and Recommendations

In this step, I create a personal movie preference profile by rating 10 movies from the MovieLens dataset. I then use the embeddings from the full dataset to generate personalized movie recommendations based on cosine similarity.

### 5a) Create Personal Movie Preference Profile

The following code creates and saves a profile that contains 10 movies that I selected and rated. These ratings are used to construct a personalized embedding profile for recommendation generation.

In [41]:
my_profile = pd.DataFrame([
    {"Title": "Toy Story (1995)", "My_Rating": 5},
    {"Title": "Clueless (1995)", "My_Rating": 5},
    {"Title": "Titanic (1997)", "My_Rating": 4},
    {"Title": "Jurassic Park (1993)", "My_Rating": 5},
    {"Title": "Men in Black (1997)", "My_Rating": 4},
    {"Title": "Beauty and the Beast (1991)", "My_Rating": 5},
    {"Title": "You've Got Mail (1998)", "My_Rating": 5},
    {"Title": "Good Will Hunting (1997)", "My_Rating": 4},
    {"Title": "Mrs. Doubtfire (1993)", "My_Rating": 5},
    {"Title": "Aladdin (1992)", "My_Rating": 5},
])

my_profile = my_profile.merge(all_movies, on="Title", how="left")
my_profile

,Title,My_Rating,MovieID,Genres,Text
0,Toy Story (1995),5,1,Animation|Children's|Comedy,Toy Story (1995) Genres: Animation|Children's|...
1,Clueless (1995),5,39,Comedy|Romance,Clueless (1995) Genres: Comedy|Romance
2,Titanic (1997),4,1721,Drama|Romance,Titanic (1997) Genres: Drama|Romance
3,Jurassic Park (1993),5,480,Action|Adventure|Sci-Fi,Jurassic Park (1993) Genres: Action|Adventure|...
4,Men in Black (1997),4,1580,Action|Adventure|Comedy|Sci-Fi,Men in Black (1997) Genres: Action|Adventure|C...
5,Beauty and the Beast (1991),5,595,Animation|Children's|Musical,Beauty and the Beast (1991) Genres: Animation|...
6,You've Got Mail (1998),5,2424,Comedy|Romance,You've Got Mail (1998) Genres: Comedy|Romance
7,Good Will Hunting (1997),4,1704,Drama,Good Will Hunting (1997) Genres: Drama
8,Mrs. Doubtfire (1993),5,500,Comedy,Mrs. Doubtfire (1993) Genres: Comedy
9,Aladdin (1992),5,588,Animation|Children's|Comedy|Musical,Aladdin (1992) Genres: Animation|Children's|Co...


In [42]:
my_profile.to_csv("my_user_profile.csv", index=False)

s3.upload_file(
    "my_user_profile.csv",
    BUCKET,
    "outputs/my_user_profile.csv"
)

print("Saved my user profile to S3")

Saved my user profile to S3


### Generate Recommendations from My Profile

This section creates and saves personalized movie recommendations based on my selected movie ratings. Movies rated 4 or higher are treated as liked movies and used to build a user preference embedding. Cosine similarity is then calculated between my preference embedding and all movie embeddings in the dataset. Movies already included in my profile are removed so that only new recommendations are returned.

In [43]:
liked = my_profile[my_profile["My_Rating"] >= 4]

liked_indices = liked.index.tolist()

my_embedding = all_embeddings[liked_indices].mean(axis=0).reshape(1, -1)

scores = cosine_similarity(my_embedding, all_embeddings)[0]

all_scored = all_movies.copy()
all_scored["score"] = scores

seen_titles = set(my_profile["Title"])
all_scored = all_scored[~all_scored["Title"].isin(seen_titles)]

my_recs = all_scored.sort_values("score", ascending=False).head(5)

my_recs[["Title", "Genres", "score"]]

,Title,Genres,score
71,Kicking and Screaming (1995),Comedy|Drama,0.790401
37,It Takes Two (1995),Comedy,0.788328
649,Mutters Courage (1995),Comedy,0.785827
2977,Incredibly True Adventure of Two Girls in Love...,Comedy|Romance,0.778802
173,Kids (1995),Drama,0.774911


In [44]:
my_recs.to_csv("my_recommendations.csv", index=False)

s3.upload_file(
    "my_recommendations.csv",
    BUCKET,
    "outputs/my_recommendations.csv"
)

print("Saved my recommendations to S3")

Saved my recommendations to S3
